# Notebook 1: Downloading GSE181919 Dataset

This notebook downloads the single-cell RNA-seq dataset **GSE181919** from NCBI GEO,
used in:

> Jarwal et al. (2024) *A deep learning method for classification of HNSCC and HPV patients using single-cell transcriptomics.* Frontiers in Molecular Biosciences.

The dataset contains 37 tissue specimens from 23 HNSCC patients sequenced on Illumina HiSeq 4000 (10x Genomics scRNA-seq).

**What this notebook does:**
1. Downloads the raw count matrices (MEX format: `matrix.mtx.gz`, `barcodes.tsv.gz`, `features.tsv.gz`) for all samples.
2. Downloads sample metadata so we can identify Normal vs. Primary Cancer and HPV+/HPV− status.
3. Organises everything into a `data/` directory.

In [ ]:
# Install dependencies if needed
# !pip install GEOparse requests pandas

In [ ]:
import os
import requests
import tarfile
import gzip
import shutil
import pandas as pd

DATA_DIR = "/home/anupam/HNSCPred/data"
RAW_DIR = os.path.join(DATA_DIR, "raw")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Data will be stored in: {os.path.abspath(DATA_DIR)}")

## 1. Download the supplementary tar file from GEO

GSE181919 provides the 10x CellRanger output files as a single tar archive.
Each sample has its own folder with `matrix.mtx.gz`, `barcodes.tsv.gz`, and `features.tsv.gz`.

In [ ]:
# The GEO supplementary file URL for GSE181919
GEO_SUPP_URL = "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE181919&format=file"

tar_path = os.path.join(RAW_DIR, "GSE181919_RAW.tar")

if not os.path.exists(tar_path):
    print("Downloading GSE181919 supplementary files (this may take a while)...")
    resp = requests.get(GEO_SUPP_URL, stream=True)
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))
    downloaded = 0
    with open(tar_path, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024*1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                print(f"\r  {downloaded/(1024**2):.0f} / {total/(1024**2):.0f} MB", end="")
    print("\nDownload complete.")
else:
    print(f"Already downloaded: {tar_path}")

In [ ]:
# Extract the tar archive
extract_dir = os.path.join(RAW_DIR, "extracted")
if not os.path.exists(extract_dir):
    print("Extracting tar archive...")
    os.makedirs(extract_dir, exist_ok=True)
    with tarfile.open(tar_path, 'r') as tar:
        tar.extractall(path=extract_dir)
    print(f"Extracted to {extract_dir}")
else:
    print("Already extracted.")

# List what we got
extracted_files = os.listdir(extract_dir)
print(f"\nNumber of files extracted: {len(extracted_files)}")
for f in sorted(extracted_files)[:10]:
    print(f"  {f}")
print("  ...")

## 2. Build sample metadata

From the GEO page and the original paper (Choi et al., 2023), the 29 samples used in the study are:
- **9 Normal** tissue samples
- **20 Primary HNSCC** samples (13 HPV−, 7 HPV+)

The metadata below is reconstructed from the GEO series matrix and the paper's supplementary information.

In [ ]:
# You can also programmatically parse the series matrix file.
# Here we provide the curated metadata from GEO for the 29 samples used in the paper.

# Download the series matrix to extract metadata
SERIES_MATRIX_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE181nnn/GSE181919/matrix/"
    "GSE181919_series_matrix.txt.gz"
)

matrix_path = os.path.join(RAW_DIR, "GSE181919_series_matrix.txt.gz")
if not os.path.exists(matrix_path):
    print("Downloading series matrix...")
    resp = requests.get(SERIES_MATRIX_URL)
    resp.raise_for_status()
    with open(matrix_path, 'wb') as f:
        f.write(resp.content)
    print("Done.")
else:
    print("Series matrix already downloaded.")

In [ ]:
# Parse sample characteristics from the series matrix
import gzip

sample_titles = []
sample_chars = {}
geo_accessions = []

with gzip.open(matrix_path, 'rt') as f:
    for line in f:
        line = line.strip()
        if line.startswith('!Sample_title'):
            parts = line.split('\t')[1:]
            sample_titles = [p.strip('"') for p in parts]
        elif line.startswith('!Sample_geo_accession'):
            parts = line.split('\t')[1:]
            geo_accessions = [p.strip('"') for p in parts]
        elif line.startswith('!Sample_characteristics_ch1'):
            parts = line.split('\t')[1:]
            values = [p.strip('"') for p in parts]
            # Extract the key from the first value
            if ':' in values[0]:
                key = values[0].split(':')[0].strip()
                if key not in sample_chars:
                    sample_chars[key] = []
                sample_chars[key].append([v.split(':', 1)[-1].strip() for v in values])

print(f"Found {len(sample_titles)} samples")
print(f"Characteristic keys: {list(sample_chars.keys())}")

In [ ]:
# Build a metadata dataframe
# The exact characteristic keys may vary; adapt as needed after inspecting the output above.

meta_records = []
for i, title in enumerate(sample_titles):
    record = {
        'geo_accession': geo_accessions[i],
        'title': title,
    }
    for key, val_lists in sample_chars.items():
        # val_lists may have multiple rows for the same key
        for row_idx, val_list in enumerate(val_lists):
            col_name = key if row_idx == 0 else f"{key}_{row_idx+1}"
            record[col_name] = val_list[i]
    meta_records.append(record)

metadata = pd.DataFrame(meta_records)
print(metadata.shape)
metadata.head(10)

In [ ]:
# Filter to the 29 samples used in the paper: Normal (n=9) + Primary cancer (n=20)
# Look for the tissue_type or similar column. Adjust column names based on the output above.

print("All columns:", metadata.columns.tolist())
print("\nUnique values per column:")
for col in metadata.columns:
    if col not in ['geo_accession', 'title']:
        print(f"  {col}: {metadata[col].unique()[:10]}")

In [ ]:
# Save metadata
metadata.to_csv(os.path.join(DATA_DIR, "GSE181919_metadata.csv"), index=False)
print(f"Metadata saved to {DATA_DIR}/GSE181919_metadata.csv")
print(f"\nTotal samples: {len(metadata)}")
print("\nYou should now filter for 'Normal' and 'Primary cancer' tissue types")
print("and identify HPV+/HPV- status from the metadata columns above.")

## 3. Verify the download

Each sample should have three files (10x CellRanger output):
- `barcodes.tsv.gz`
- `features.tsv.gz` (or `genes.tsv.gz`)
- `matrix.mtx.gz`

In [ ]:
# Check the extracted files
print(f"Files in {extract_dir}:")
for f in sorted(os.listdir(extract_dir)):
    size_mb = os.path.getsize(os.path.join(extract_dir, f)) / (1024**2)
    print(f"  {f}  ({size_mb:.1f} MB)")

## Next Steps

Proceed to **Notebook 2** (`02_hnscc_classification.ipynb`) which:
1. Loads & merges the count matrices using `scanpy`
2. Pre-processes (filtering, CPM normalisation, log-transform)
3. Performs mRMR feature selection (top 100 genes)
4. Trains ML models (DT, RF, LR, XGB, ET, KNN) and the ANN deep learning model
5. Evaluates on the 20% validation set
6. Performs HPV+/HPV− classification
7. Runs Gene Ontology enrichment analysis